I've been working on decoding the sub-graph of a simple MNIST model.  

The major outline of the approach is:
- Use attribution techniques to find a subgraph which is important, to cut the surface area of checking
- Prune the attribution graph by finding points which are high contributing (cuts the surface area a bit more)
- Investigate what a convolution is doing, assign a label to every distinct action that the convolution is doing (finding a diagonal? finding a vertical line?).  
- the core construct right is that each number in the internal activation has a meaning, how that meaning is calculated is not very relevant (it could be a combination of simpler logical propositions). A convolutional kernel is a very basic transformer (its really just a linear transformation lol.) which adds onto meaning.   
  - A convolutional kernel might say: "The pixels on my left column are red, the ones on the center and right are green". This is a way to say "tis vertical man". But the kernel does not know that framing. A big problem with investigating convolutional networks is that it is too easy to say something very concrete for many operations (in earlier layers of bigger networks, in all layers of a small mnist network).
  - Concretely vocalising what a kernel is doing breaks the actual abstraction: It lets a set of patterns to pass through. I want to build from this very basic definition eventually.  
- The important part right now, is assigning a set of labels to the current convolutional operation. Each label is a single pure pattern that the kernel passed through.  


Let's see where that gets me.  


# Assigning meaning to a transformation

When we look at a 2D convolutional operation, it is very apparent on what the operation is doing. We say it out loud, like "it is finding a vertical edge". What really is this statement? The statement captures a "meaning" in the input image, what is meaning though?  
After some thought, the answer is quite simple. A meaning is a pattern consistently occurring, given a name.  

So the task is to now find patterns in a convolutional kernel. If we are able to annotate what is happening at each activation, we are closer to understanding what the model is really doing. I call this labelling an operation. We look very closely at each operation, maximising squiting, and hopefully we'll be able to assign a label to what the kernel is doing.  
Note that, we don't assign a label to the whole activation (say a 7x7 output of a kernel). That is not enough, we need to do it for EVERY number in that 7x7 output activation, a total of 49 numbers. This is basically the total number of "raw numbers" in the intermediate states between layers. To assign a label to a single output activation number, you look at the 3x3 input patch and the kernel.   

> Although not relevant to this discussion, practically, just looking at a 3x3 input patch is not enough.   
> You need to look at the surrounding. The reason is you would find many examples where if you zoom out a bit, you see that the output number you are analysing is part of a diagonal that a kernel is passing through.  
> Most of the times, the input patch would be predictable. Many times though, it would just look weird. Junk. Extremely different than the normal patterns found (it won't even look like the kernel is detecting a diagonal if you conventionally just look at the patch).  
> Is this a coincidence? I'm not sure, maybe gradient descent just overfit the definition of the diagonal to those kinds of patches? There's not enough information to make a statement in this. Investigating and catering to this phenomenon is out of the scope of the current approach though, these are considered outliers for now.  

The labels we assign are generally imperfect. If its a diagonal, what is the angle of it? What is the intensity? Many times, simply showing the zoomed out photo with the patch of the input activation is very useful (the picture contains a lot more information than words). Still, its not very exact. It's also very hard to work with. We can't work on the data at a higher level without assigning something concrete.    

So we start out by manually labelling each output activation. This is a very manual, boring and painful task. It gets easier to understand what a certain kind of kernel is doing with time, but the sheer diversity of kernels is overwhelming. I suspect there might be a huge learning curve every time in understanding the exact mechanism of a kernel in detail.    
But this is not a very scalable approach. I still went ahead with it to get a feel of it.  

It's time to automate it. It would be beautiful if I just ran my model with a given input, and it gave me a graph of labels for that run. The circuit of that run. This would be an unsupervised learning experiment. We want to group our input activations, the first approach that comes to mind is clustering. This should be easy enough? Wrong. They don't cluster well (even the simplest 2D case).  
I've tried HDBScan, K-Means, etc. With/without dimensionality reduction. Nothing seems to cluser nicely. I was stuck for a while thinking that I don't currently have the skills to work with a lot of data. 


In [ ]:
# TODO: add a clustering example and show it is behaving for random points

The problem however, was noise, a convolutional kernel generally does not allow most of the patterns to pass through. If you look at a 3x3 convolutional kernel, it has 9 indepdent numbers, thats a lot of combinations of patterns. Out of it, it allows a very small subset.  
It only makes sense to label these small subset of patterns. After all, this is what we do manually to. In practice, if the input pattern is noise to the kernel, the output activation is a very small number. In a way, this pattern is rejected by the kernel. It terminates this branch of the subgraph. This is why we want to only cluster this very small subset.   

How can we judge if a pattern has semantic meaning to the kernel? The easiest way is to find all the high activations of a kernel (positive or negative) and group on them. I'm not sure if this is good enough. The approach I went with is finding all examples whose outputs had high attribution. A higher attribution score means that the downstream layers felt that the output activation in question, is useful. This is our whole assumption, that the circuit of maximal usefulness has semantic meaning at every step. So I went with this approach.  

Considering the pains I had with clustering before, I restricted the problem even more. We just focus on a single output number. A convolutional kernel at some layer, lets say, gives an output of dimension `7x7` for some input patch, we focus only one on of the coordinates inside this number, say `(2,3)`.  

A convolution operation can simply be written as `Sum(w[i] * x[i])`. We can cluser on `x[i]`. Or we can cluster on `w[i]*x[i]`.  
A reason I clustered directly on these pointwise multiplications was my paranoia that clustering won't work. The other reason is that if `w[i]` is `0` or near `0` at some `i`, then every `x[i]` at that position is just noise to clustering.  

In [ ]:
# TODO: add a clustering example show it is behaving for high POIs

# A Conv2d kernel is 3D in shape

I was reasonably happy with the clustering, and I had started on trying to analyse patterns across kernels now. The drill is simple, you find the subgraph of maximal activation using some attribution technique, and look at a single circuit of an output pixel. With just 2 layers, you can make a UI and the patterns are extremely obvious.  

A convolutional layer is of the shape `[out_channels, in_channels, H, W]`, where you have `out_channel` kernels, each kernel of size `[in_channels, H, W]`. I was analysing each of the `in_channels` 2D kernels `H,W` as simple 2D convolutional operations.  
The problem is that it is a very human interpretation. I started out because I thought convolutional networks are more intuitive to our eyes, which is true for some extent. But generally, its a lie.  
Gradient descent does not care about which set of patterns group together. You can also analyse a kernel vertically. Or diagonally. Or just ask your baby nephew to point to a direction and start hustling.  

This realisation was a kind of hit to me. I had spent the last two months coming up with an attribution method, closely looking at each kernel, being fascinated by some of the patterns I found.  
We need a new way to look at this.  


## Abstract it out

I was concerned from the beginning that the deeper layer of convolutional kernels might not even look like they are detecting a "visual" pattern.  
I imagined them as simply kernels which pass a certain pattern of numbers. The pattern is arbitrary, only dependent on how they line up with gradient descent. This is just speculation btw. In any case, the visual approach has its limits now.   
Simply looking at MNIST might have been a mistake, but I was kinda scared of starting with a ResNet.     

The core problem is simple, a kernel allows a set of patterns to pass through. Let's say it allows the labels `{A, B, C}`. A 3D convolutional kernel is huge. It can pass through multiple patterns. It can pass them in any combination. In some input it could be `{A, B}`, or `{A, B, C}` or `{B, C}`.   
We can frame it as finding a set of basis vectors in the space of all patterns the kernel allows. We want to extract `A`, `B` and `C` in separate vectors and write the input vector as a linear combination of these three basis vectors.   

Claude :) suggested using Dictionary Learning. Generally, we can also study about independent component analysis.  

### Dictionary learning

Dictionary learning basically trains a model to provide a weight matrix `W` and a set of vector `S`, `W` is your basis (decomposed patterns) stacked in a matrix. `S` stores a row for each input example, where each row contains coefficients for each basis vector in `W`.  
There are two very nice properties about dictionary learning:
- It is unsupervised. It is basically like an autoencoder, it provides us with 2 matrices which regenerate the input.  
- It gives sparse coefficients

Since it gives sparse coefficients, you can assume that a lot of your data would be a linear combination of a small number of basis vectors.   
You can increase sparsity if you want by tuning its hyperparameters. For the MNIST model, I got a reasonably simple model which had a lot of sparse coefficients where only one basis vector was active.  


The implications are quite useful. If a lot of your input can be expressed by a single basis vector (multiplied by some scalar value), then your basis vectors are essentially the combinations of your individual groups found statistically in your dataset.  

A single basis vector might be the combination of patterns `{A, B, C}`, while the other might be `{B, C}`.   
This is a good step towards assigning labels now. It also has some really non-intuitive properties which blew me away.

- The basis vectors are very stable across different random initialisations (this is a very good sign).
- The reconstructed vectors are also quite good, they match my dataset.   


For the clustering example in 2D, they work. They are also thankfully working in 3D :).  
The problem with clustering in 3D is that I don't know the structure of my data at all. It requires a lot of trial and error to come up with a strategy. This does not scale at all.  

It seems we can maybe now assign labels to our activations automagically, yay. The subgraph would now be annotated, allowing us to look at it at a more higher level and work with it. There is another good thing about this. Attribution methods are famously known for lying. Two attribution methods can give very different results and would both sound credible at the same time.  
If we assume that attribution techniques give us points which cluster in a stable way, we can use as many attribution techniques as we want to find sample data. Each attribution method has its own weaknesses, by using them all, we try to cover as much ground as we can.  
The saliency map is now simply the path we were able to track, the path where none of the kernels said that the input pattern is garbage.  
Although we can't assign importances yet, this is a good first step.   